In [16]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# =========================
# FILE PATH
# =========================
file_path = r'C:\Users\Users\pharmaceuticalSupplyChain.csv'
encoding = 'latin-1'

# =========================
# CARD STYLE FUNCTION
# =========================
def card_style():
    return {
        'padding': '20px',
        'border': '1px solid #e0e0e0',
        'borderRadius': '12px',
        'backgroundColor': '#ffffff',
        'width': '30%',
        'textAlign': 'center',
        'boxShadow': '0 2px 8px rgba(0,0,0,0.08)',
        'transition': 'transform 0.2s ease'
    }

# =========================
# READ FILE LINES
# =========================
with open(file_path, 'r', encoding=encoding) as f:
    lines = f.readlines()

disruptions_start = None
shipping_start = None

for i, line in enumerate(lines):
    if 'date,year,event_name,disruption_type' in line:
        disruptions_start = i
    elif 'date,year,month,baltic_dry_index,container_rate_usd_40ft' in line:
        shipping_start = i

print("Disruptions start:", disruptions_start)
print("Shipping start:", shipping_start)

# =========================
# LOAD PHARMA DATA
# =========================
pharma_df = pd.read_csv(
    file_path,
    nrows=disruptions_start - 1,
    encoding=encoding
)

pharma = pharma_df[
    pharma_df['industry'] == 'Pharma & Medical'
].copy()

# ===== FIX: Convert to numeric FIRST, then fix the corrupted value =====
numeric_cols = [
    'pandemic_exposure',
    'geopolitical_exposure',
    'natural_disaster_exposure',
    'tariff_exposure',
    'logistics_exposure',
    'overall_vulnerability',
    'inventory_days'
]

# First convert all numeric columns (this will turn the corrupted 'e' to NaN)
for col in numeric_cols:
    pharma[col] = pd.to_numeric(pharma[col], errors='coerce')

# THEN fix the corrupted value in 2002
pharma.loc[pharma['year'] == 2002, 'geopolitical_exposure'] = 5.6

print("Pharma rows:", len(pharma))

# =========================
# LOAD DISRUPTIONS DATA
# =========================
disruptions = pd.read_csv(
    file_path,
    skiprows=disruptions_start,
    nrows=shipping_start - disruptions_start - 1,
    encoding=encoding
)

# Clean columns
disruptions.columns = disruptions.columns.str.strip()

# Remove duplicated header row if exists
if 'date' in disruptions.columns:
    disruptions = disruptions[
        disruptions['date'] != 'date'
    ].copy()

if 'freight_rate_shock_pct' in disruptions.columns:
    disruptions['freight_rate_shock_pct'] = pd.to_numeric(
        disruptions['freight_rate_shock_pct'],
        errors='coerce'
    )

print("Disruptions rows:", len(disruptions))

# =========================
# LOAD SHIPPING DATA
# =========================
shipping = pd.read_csv(
    file_path,
    skiprows=shipping_start,
    encoding=encoding,
    header=0
)

# Clean columns
shipping.columns = shipping.columns.str.strip()

# Remove duplicated header row
if 'date' in shipping.columns:
    shipping = shipping[
        shipping['date'] != 'date'
    ].copy()

# Convert numeric
if 'container_rate_usd_40ft' in shipping.columns:
    shipping['container_rate_usd_40ft'] = pd.to_numeric(
        shipping['container_rate_usd_40ft'],
        errors='coerce'
    )

# Convert date
if 'date' in shipping.columns:
    shipping['date'] = pd.to_datetime(
        shipping['date'],
        errors='coerce'
    )

# Drop invalid rows
shipping = shipping.dropna(
    subset=['container_rate_usd_40ft']
)

print("Shipping rows:", len(shipping))

# =========================
# CUSTOM COLOR PALETTE
# =========================
# Main dashboard colors
COLORS = {
    'primary': '#1f77b4',      # Blue - main trend lines
    'secondary': '#ff7f0e',    # Orange - secondary elements
    'success': '#2ca02c',      # Green - positive/normal
    'danger': '#d62728',       # Red - high risk/crisis
    'warning': '#ffbb78',      # Light orange - thresholds
    'purple': '#9467bd',       # Purple - exposure chart
    'teal': '#17becf',         # Teal - disruptions
    'bg': '#f7f9fc',           # Light background
    'card_bg': '#ffffff',      # Card background
    'text_muted': '#7f8c8d',   # Muted text
    'text_dark': '#2c3e50'     # Dark text
}

# Color scales for different chart types
EXPOSURE_COLOR_SCALE = ['#fee5d9', '#fcae91', '#fb6a4a', '#de2d26', '#a50f15']
DISRUPTION_COLOR_SCALE = ['#2ca02c','#90ee90','#ffd166','#ff6b6b','#d62728']

# =========================
# CREATE DASH APP
# =========================
app = Dash(__name__)

app.layout = html.Div(
    style={
        'fontFamily': 'Segoe UI, Arial, sans-serif',
        'backgroundColor': COLORS['bg'],
        'minHeight': '100vh',
        'padding': '30px'
    },
    children=[

    html.H1(
        "Pharmaceutical Supply Chain Dashboard",
        style={
            'textAlign': 'center',
            'color': COLORS['text_dark'],
            'marginBottom': '30px',
            'fontWeight': '600',
            'fontSize': '28px'
        }
    ),

    # KPI ROW
    html.Div([
        html.Div(id='kpi-vuln', style=card_style()),
        html.Div(id='kpi-freight', style=card_style()),
        html.Div(id='kpi-inventory', style=card_style())
    ], style={
        'display': 'flex',
        'justifyContent': 'space-around',
        'marginBottom': '30px',
        'gap': '20px'
    }),

    # Vulnerability Trend
    dcc.Graph(id='vulnerability-chart'),

    # Two charts side by side
    html.Div([
        dcc.Graph(
            id='exposure-chart',
            style={'width': '48%', 'display': 'inline-block'}
        ),
        dcc.Graph(
            id='freight-chart',
            style={'width': '48%', 'display': 'inline-block'}
        )
    ], style={
        'display': 'flex',
        'justifyContent': 'space-between',
        'marginBottom': '30px'
    }),

    # Disruption chart
    dcc.Graph(id='disruption-chart')

])

# =========================
# CALLBACK
# =========================
@app.callback(
    [
        Output('vulnerability-chart', 'figure'),
        Output('exposure-chart', 'figure'),
        Output('freight-chart', 'figure'),
        Output('disruption-chart', 'figure'),
        Output('kpi-vuln', 'children'),
        Output('kpi-freight', 'children'),
        Output('kpi-inventory', 'children')
    ],
    Input('vulnerability-chart', 'id')
)
def update_dashboard(_):
    # Safety check for empty dataframes
    if len(pharma) == 0:
        empty_fig = px.line(title='No data available')
        empty_kpi = html.Div([html.H2("No Data"), html.P("Available")])
        return (empty_fig, empty_fig, empty_fig, empty_fig, 
                empty_kpi, empty_kpi, empty_kpi)
    
    latest_year = pharma['year'].max()
    latest = pharma[pharma['year'] == latest_year].iloc[0]

    # =====================
    # KPI 1 - Vulnerability
    # =====================
    peak_vuln = pharma['overall_vulnerability'].max()
    vuln_kpi = html.Div([
        html.H2(
            f"{peak_vuln:.2f}",
            style={
                'color': COLORS['danger'],
                'margin': '0',
                'fontSize': '32px',
                'fontWeight': '700'
            }
        ),
        html.P(
            "Peak Vulnerability Score",
            style={
                'margin': '10px 0 0 0',
                'color': COLORS['text_muted'],
                'fontSize': '14px'
            }
        ),
        html.Small(
            f"Year: {int(pharma[pharma['overall_vulnerability'] == peak_vuln]['year'].iloc[0])}",
            style={'color': COLORS['text_muted'], 'fontSize': '12px'}
        )
    ])

    # =====================
    # KPI 2 - Freight Rate
    # =====================
    if len(shipping) > 0:
        peak_rate = shipping['container_rate_usd_40ft'].max()
        peak_date = shipping.loc[shipping['container_rate_usd_40ft'].idxmax(), 'date'].strftime('%Y-%m')
        freight_kpi = html.Div([
            html.H2(
                f"${peak_rate:,.0f}",
                style={
                    'color': COLORS['secondary'],
                    'margin': '0',
                    'fontSize': '32px',
                    'fontWeight': '700'
                }
            ),
            html.P(
                "Peak Container Rate",
                style={
                    'margin': '10px 0 0 0',
                    'color': COLORS['text_muted'],
                    'fontSize': '14px'
                }
            ),
            html.Small(
                f"Date: {peak_date}",
                style={'color': COLORS['text_muted'], 'fontSize': '12px'}
            )
        ])
    else:
        freight_kpi = html.Div([html.H2("No Data"), html.P("Available")])

    # =====================
    # KPI 3 - Inventory Days
    # =====================
    inventory_days = int(latest['inventory_days']) if not pd.isna(latest['inventory_days']) else 0
    inventory_kpi = html.Div([
        html.H2(
            f"{inventory_days} Days",
            style={
                'color': COLORS['primary'],
                'margin': '0',
                'fontSize': '32px',
                'fontWeight': '700'
            }
        ),
        html.P(
            "Inventory Buffer",
            style={
                'margin': '10px 0 0 0',
                'color': COLORS['text_muted'],
                'fontSize': '14px'
            }
        ),
        html.Small(
            "Pharma industry average",
            style={'color': COLORS['text_muted'], 'fontSize': '12px'}
        )
    ])

    # =====================
    # CHART 1 - Vulnerability Trend
    # =====================
    fig1 = px.line(
        pharma,
        x='year',
        y='overall_vulnerability',
        markers=True,
        title='Overall Vulnerability Trend (2000-2024)',
        color_discrete_sequence=[COLORS['primary']]
    )

    fig1.add_hline(
        y=5,
        line_dash="dash",
        line_color=COLORS['warning'],
        annotation_text="High Risk Threshold",
        annotation_position="top right"
    )

    fig1.update_layout(
        template='plotly_white',
        hovermode='x unified',
        title_font=dict(size=16, color=COLORS['text_dark']),
        font=dict(family="Segoe UI, Arial", size=12),
        plot_bgcolor=COLORS['bg'],
        paper_bgcolor=COLORS['bg'],
        xaxis=dict(title="Year", gridcolor='#e0e0e0'),
        yaxis=dict(title="Vulnerability Score (0-10)", gridcolor='#e0e0e0', range=[4, 7])
    )

    fig1.update_traces(
        marker=dict(size=8, color=COLORS['primary'], line=dict(width=1, color='white')),
        line=dict(width=2)
    )

    # =====================
    # CHART 2 - Exposure by Type
    # =====================
    exposure_df = pd.DataFrame({
        'Type': ['Pandemic', 'Geopolitical', 'Logistics', 'Natural Disaster', 'Tariff'],
        'Score': [
            latest['pandemic_exposure'] if not pd.isna(latest['pandemic_exposure']) else 0,
            latest['geopolitical_exposure'] if not pd.isna(latest['geopolitical_exposure']) else 0,
            latest['logistics_exposure'] if not pd.isna(latest['logistics_exposure']) else 0,
            latest['natural_disaster_exposure'] if not pd.isna(latest['natural_disaster_exposure']) else 0,
            latest['tariff_exposure'] if not pd.isna(latest['tariff_exposure']) else 0
        ]
    })

    fig2 = px.bar(
        exposure_df,
        x='Type',
        y='Score',
        color='Score',
        title=f'Exposure Breakdown ({latest_year})',
        color_continuous_scale=EXPOSURE_COLOR_SCALE,
        text='Score'
    )

    fig2.update_traces(
        textposition='outside',
        textfont=dict(size=12, color=COLORS['text_dark'])
    )

    fig2.update_layout(
        template='plotly_white',
        title_font=dict(size=16, color=COLORS['text_dark']),
        font=dict(family="Segoe UI, Arial", size=12),
        plot_bgcolor=COLORS['bg'],
        paper_bgcolor=COLORS['bg'],
        xaxis=dict(title="", gridcolor='#e0e0e0'),
        yaxis=dict(title="Exposure Score (0-10)", gridcolor='#e0e0e0', range=[0, 12])
    )

    # =====================
    # CHART 3 - Freight Rates
    # =====================
    if len(shipping) > 0:
        shipping_sorted = shipping.sort_values('date')

        fig3 = px.line(
            shipping_sorted,
            x='date',
            y='container_rate_usd_40ft',
            title='Container Freight Rates (2000-2024)',
            color_discrete_sequence=[COLORS['success']]
        )

        # Add colored zones for context
        fig3.add_hrect(y0=0, y1=2000, line_width=0, fillcolor="green", opacity=0.1, annotation_text="Normal")
        fig3.add_hrect(y0=2000, y1=5000, line_width=0, fillcolor="yellow", opacity=0.1, annotation_text="Elevated")
        fig3.add_hrect(y0=5000, y1=14000, line_width=0, fillcolor="red", opacity=0.1, annotation_text="Crisis")

        fig3.update_layout(
            template='plotly_white',
            hovermode='x unified',
            title_font=dict(size=16, color=COLORS['text_dark']),
            font=dict(family="Segoe UI, Arial", size=12),
            plot_bgcolor=COLORS['bg'],
            paper_bgcolor=COLORS['bg'],
            xaxis=dict(title="Date", gridcolor='#e0e0e0'),
            yaxis=dict(title="USD per 40ft Container", gridcolor='#e0e0e0')
        )

        fig3.update_traces(line=dict(width=2))

        # Add peak annotation
        fig3.add_annotation(
            x=shipping.loc[shipping['container_rate_usd_40ft'].idxmax(), 'date'],
            y=peak_rate,
            text=f"Peak: ${peak_rate:,.0f}",
            showarrow=True,
            arrowhead=2,
            arrowcolor=COLORS['secondary'],
            font=dict(size=11, color=COLORS['secondary'])
        )
    else:
        fig3 = px.line(title='No freight rate data available')

    # =====================
    # CHART 4 - Disruption Events
    # =====================
    if len(disruptions) > 0 and 'freight_rate_shock_pct' in disruptions.columns:
        disruptions_clean = disruptions.dropna(subset=['freight_rate_shock_pct'])
        
        if len(disruptions_clean) > 0:
            top_events = disruptions_clean.nlargest(10, 'freight_rate_shock_pct')

            fig4 = px.bar(
                top_events,
                x='freight_rate_shock_pct',
                y='event_name',
                orientation='h',
                color='freight_rate_shock_pct',
                title='Top 10 Disruption Events by Freight Rate Impact',
                color_continuous_scale=DISRUPTION_COLOR_SCALE,
                text='freight_rate_shock_pct'
            )

            fig4.update_traces(
                texttemplate='%{text}%',
                textposition='outside',
                textfont=dict(size=11)
            )

            fig4.update_layout(
                template='plotly_white',
                title_font=dict(size=16, color=COLORS['text_dark']),
                font=dict(family="Segoe UI, Arial", size=12),
                plot_bgcolor=COLORS['bg'],
                paper_bgcolor=COLORS['bg'],
                xaxis=dict(title="Freight Rate Change (%)", gridcolor='#e0e0e0'),
                yaxis=dict(title="", gridcolor='#e0e0e0', categoryorder='total ascending'),
                height=500,
                margin=dict(l=250, r=50, t=50, b=50)
            )
        else:
            fig4 = px.bar(title='No disruption events with rate impact')
    else:
        fig4 = px.bar(title='Disruption data not available')

    return (fig1, fig2, fig3, fig4, vuln_kpi, freight_kpi, inventory_kpi)

# =========================
# RUN APP
# =========================
if __name__ == '__main__':
    print("=" * 60)
    print("Dashboard running at:")
    print("http://127.0.0.1:8050")
    print("=" * 60)
    app.run(debug=True)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Users\\pharmaceuticalSupplyChain.csv'